<a href="https://colab.research.google.com/github/T-Chiunzi/Tava-Smart-Finance-Assistant/blob/main/starter_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pseudocode

Send greetings to the user

Display Dashboard

Ask the user to input their salary

Ask user what they want to save for and how much of their salary they want to dedicate to it

User inputs their savings goals or 0 to stop

Display to the user that they should input their expenses

Ask the user if they want to input their expenses via CSV or manually.

If via CSV, run it

If manually, ask the user to input expenses or 0 to stop

Read the data

Output table of expenses

Output a pie chart of expenses

If expenses are more than what they want to save, then advise which expenses they should reduce

Ask the user for how long they want to save for their savings goal by months or indefinitely (for emergency savings)

Ask the user if they want to increase the amount they save by (by % or by money)

Display a timeline of savings with visual of a person running along the line


In [1]:
!pip install gradio pandas hands-on-ai
import os

os.environ['HANDS_ON_AI_SERVER'] = 'https://ollama.serveur.au'
os.environ['HANDS_ON_AI_MODEL'] = 'llama3.2'
os.environ['HANDS_ON_AI_API_KEY'] = 'isys2001-assignment-key'

print("✅ 🔑 Hands-on-AI configured successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 20.7 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.14.0
    Uninstalling jiter-0.14.0:
      Successfully uninstalled jiter-0.14.0
✅ 🔑 Hands-on-AI configured successfully!


In [2]:
from hands_on_ai.chat import get_response

try:
    # Testing the precise function verified in your class handout
    test_response = get_response("Hello! This is a connection handshake test.")
    print("🚀 Hands-on-AI connection successful!")
    print(f"Response from campus server: {test_response}")
except Exception as e:
    print(f"❌ Connection issue still persisting: {e}")

⚙️ Switching to model: llama3.2 ... (may take a few seconds)


[WARNING] Error during request (attempt 1): Connection error.


Oops! I got a little tangled up... Let's try that again 😊


[WARNING] Error during request (attempt 2): Connection error.


🚀 Hands-on-AI connection successful!
Response from campus server: ❌ Error: Connection error.


In [ ]:
import os
import gradio as gr
import pandas as pd
from datetime import datetime

# Import the exact package frameworks from your university starter notebook
try:
    from hands_on_ai.chat import get_response
    from hands_on_ai.rag import create_index
    from hands_on_ai.agents import create_agent
except ImportError:
    pass

# Global handle to track our active RAG transaction document index
savings_rag_index = None

# =====================================================================
# 🤖 AGENT TOOL MODULE (Registered Financial Calculation Engine)
# =====================================================================

def calculate_savings_feasibility(salary, emg_goal, has_secondary, sec_cost, sec_date_str, total_expenses):
    """
    Agent Tool: Computes financial dynamics using direct positional inputs.
    Returns an actionable strategic health statement for the Agent or UI container.
    """
    try:
        sal = float(salary) if salary else 0.0
        emg = float(emg_goal) if emg_goal else 0.0
        exp = float(total_expenses) if total_expenses else 0.0

        disposable_before_savings = max(0.0, sal - exp)
        secondary_monthly_target = 0.0
        months_remaining = 1

        # Parse targets for custom secondary milestone
        if has_secondary and sec_cost and sec_cost > 0 and sec_date_str:
            try:
                target_date = datetime.strptime(sec_date_str.strip(), "%Y-%m-%d").date()
                today = datetime.now().date()
                if target_date > today:
                    days_to_sec = (target_date - today).days
                    months_remaining = max(1, round(days_to_sec / 30.41))
                    secondary_monthly_target = float(sec_cost) / months_remaining
            except ValueError:
                pass

        total_monthly_savings_goal = emg + secondary_monthly_target
        final_remaining_cash = disposable_before_savings - total_monthly_savings_goal

        # Formulate structural metrics layout for the Agent context
        report = (
            f"--- Core Savings Calculations ---\n"
            f"- Outbound Expenses Tracked: ${exp:,.2f}\n"
            f"- Disposable Income Before Savings: ${disposable_before_savings:,.2f}\n"
            f"- Compulsory Emergency Allocation: ${emg:,.2f}/month\n"
        )
        if has_secondary:
            report += f"- Secondary Goal Target: ${secondary_monthly_target:,.2f}/month (Over next {months_remaining} months)\n"
        report += f"- Total Monthly Savings Target Volume: ${total_monthly_savings_goal:,.2f}\n"
        report += f"- Final Leftover Balance: ${final_remaining_cash:,.2f}\n"
        report += f"- Strategy Feasibility Check: " + ("Affordability Confirmed! Surplus cash is positive." if final_remaining_cash >= 0 else f"Deficit Warning! Over budget target parameters by ${abs(final_remaining_cash):,.2f}.")

        return report
    except Exception as e:
        return f"Error executing calculation tool metrics: {str(e)}"


# =====================================================================
# 📝 FILE CLEANING & NATIVE RAG VECTOR DATABASE INDEX GENERATOR
# =====================================================================

def process_and_index_savings_csv(file_obj):
    """
    Cleans incoming transaction lines via pandas dataframes and registers the
    document path straight into the hands-on-ai RAG framework.
    """
    global savings_rag_index

    if file_obj is None:
        return None, "⚠️ No data document detected. Please provide a valid CSV transaction record."

    try:
        df = pd.read_csv(file_obj.name)
        df.columns = [col.strip() for col in df.columns]

        if 'Amount' not in df.columns:
            return None, "⚠️ Validation Error: File structure must contain an 'Amount' column header."

        df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
        df = df.dropna(subset=['Amount'])
        df = df[df['Amount'] > 0]

        clean_path = "processed_savings_ledger.csv"
        df.to_csv(clean_path, index=False)

        try:
            savings_rag_index = create_index(clean_path)
            status_msg = f"✅ RAG Document Index Formed! Running system active."
        except Exception:
            savings_rag_index = "local_fallback_mode"
            status_msg = f"⚠️ University Server Down. System tracking running locally!"

        return df, status_msg

    except Exception as e:
        return None, f"⚠️ Error initializing RAG asset indexing structures: {str(e)}"


# =====================================================================
# 💬 CHAT GENERATOR CONTAINER (RAG Matching + Agent Loop)
# =====================================================================

def savings_buddy_agent_response(user_message, chat_history, name, currency, salary, emg_goal, has_sec, sec_cost, sec_date, active_df):
    """
    Executes an Agent dialogue query using OpenAI style dictionary format to eliminate warning output logs.
    """
    global savings_rag_index
    if not user_message:
        return "", chat_history

    username = name if name else "Friend"
    curr = currency if currency else "$"

    try:
        total_spent = float(active_df['Amount'].sum()) if active_df is not None and not active_df.empty else 0.0
    except Exception:
        total_spent = 0.0

    system_prompt = f"""You are 'Savings Buddy', a wise, encouraging, and warm AI financial counselor.
    You have direct access to a tool named 'calculate_savings_feasibility' to compute targets,
    and a specialized RAG database index containing the user's transaction records.

    Active Workspace Context Parameters:
    - User Name: {username}
    - Currency Symbol Prefered: {curr}
    - Monthly Net Base Salary: {curr}{salary}
    - Document Baseline Aggregate Spending: {curr}{total_spent}
    """

    try:
        tool_dictionary = {
            "calculate_savings_feasibility": lambda: calculate_savings_feasibility(salary, emg_goal, has_sec, sec_cost, sec_date, total_spent)
        }
        agent = create_agent(tools=tool_dictionary, system=system_prompt)
        ai_reply = agent.run(user_message, index=savings_rag_index)

    except Exception:
        calc_summary = calculate_savings_feasibility(salary, emg_goal, has_sec, sec_cost, sec_date, total_spent)
        ai_reply = f"☕ **[Local Fallback Mode Active]** Server connection limited. Here is your tool output:\n\n{calc_summary}"

    chat_history.append({"role": "user", "content": user_message})
    chat_history.append({"role": "assistant", "content": str(ai_reply)})
    return "", chat_history


# =====================================================================
# DYNAMIC VISUAL PALETTE ENGINE (Light, Dark, and Sepia UI Styles)
# =====================================================================
def apply_realtime_styles(theme_preset):
    if theme_preset == "Dark Mode (Deep Onyx)":
        bg_color, card_color, btn_color, text_color, banner_title = "#121212", "#1e1e1e", "#3a86ff", "#f5f5f5", "#ffffff"
    elif theme_preset == "Sepia Mode (Vintage Linen)":
        bg_color, card_color, btn_color, text_color, banner_title = "#f4ecd8", "#e4d3b2", "#8c6239", "#433422", "#2b1d0c"
    else:
        bg_color, card_color, btn_color, text_color, banner_title = "#ffffff", "#f8f9fa", "#007bff", "#212529", "#000000"

    style_html = f"""
    <style>
        .gradio-container, body, html {{ background-color: {bg_color} !important; color: {text_color} !important; font-family: 'Inter', sans-serif !important; }}
        .savings-banner {{ background-color: {card_color} !important; border-radius: 12px; padding: 24px; text-align: center; border: 1px solid rgba(0,0,0,0.05); }}
        .savings-banner h1 {{ color: {banner_title} !important; margin: 0; }}
        .savings-banner p {{ color: {text_color} !important; margin: 6px 0 0 0; opacity: 0.8; }}
        button.primary {{ background-color: {btn_color} !important; color: white !important; border: none !important; font-weight: bold !important; border-radius: 8px !important; }}
    </style>
    """
    return style_html, f"🎨 Theme updated seamlessly to '{theme_preset}'!"


# =====================================================================
# GRADIO APPLICATION INTERFACE CONTEXT
# =====================================================================
with gr.Blocks(title="Savings Buddy Framework") as app:
    style_injection_box = gr.HTML("")

    with gr.Row():
        gr.Markdown("## ☕ Personalize Your Savings Dashboard Environment")
    with gr.Row():
        theme_selector = gr.Radio(
            choices=["Light Mode (Crisp Minimalist)", "Dark Mode (Deep Onyx)", "Sepia Mode (Vintage Linen)"],
            value="Light Mode (Crisp Minimalist)", label="Select App Display Aesthetic"
        )
        apply_theme_btn = gr.Button("🎨 Re-render Display Theme", variant="secondary")

    gr.Markdown("---")

    with gr.Row():
        gr.HTML("""
        <div class="savings-banner">
            <h1>🐷 Your Savings Buddy Tracker</h1>
            <p>Let's map out your earnings, insulate your emergency funds, and calculate realistic paths to your dream milestones using Agent and RAG systems! 🌟</p>
        </div>
        """)

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🌟 Step 1: Initialize Your Core Demographics & Targets")
            user_name = gr.Textbox(placeholder="What name do you prefer?", label="First Name")
            user_curr = gr.Textbox(value="$", placeholder="e.g. $, €, £", label="Preferred Currency Symbol")
            user_salary = gr.Number(value=3000, label="💰 Monthly Net Salary Income")

            gr.Markdown("---")
            gr.Markdown("#### 🛡️ Emergency Fund Configuration (Compulsory Baseline)")
            emg_target = gr.Number(value=300, label="Monthly Emergency Fund Retainer Target")

        with gr.Column():
            gr.Markdown("#### ✈️ Custom Secondary Savings Goal Horizon")
            has_secondary = gr.Checkbox(label="Activate a Secondary Custom Goal Tracker?", value=False)
            sec_name = gr.Textbox(placeholder="e.g. Holiday in Botswana", label="Secondary Goal Name")
            sec_cost = gr.Number(value=5000, label="Total Target Cost Estimate")
            sec_date = gr.Textbox(value="2027-06-12", placeholder="YYYY-MM-DD", label="Target Achievement Date Cutoff")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📊 Step 2: Feed Bank Statements into RAG Document Index")
            file_upload = gr.File(file_types=[".csv"], label="Drag & Drop Statement File for RAG Vectorization")
            csv_btn = gr.Button("🚀 Rebuild RAG Vector Database Index", variant="primary")

    with gr.Row():
        with gr.Column():
            status_ticker = gr.Textbox(value="📭 Index uninitialized.", label="System Status Terminal Updates", interactive=False)
            ledger_view = gr.DataFrame(label="Active CSV Document Structure Records", value=pd.DataFrame(columns=["Amount"]))

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🦉 Step 3: Run Automation Feasibility Evaluations")
            finalize_btn = gr.Button("📊 Process Feasibility Analytics Report", variant="primary")
            report_box = gr.Markdown("✨ *Your custom calculations framework statements will populate here...*")

        with gr.Column():
            gr.Markdown("### 💬 Step 4: Consult Your Dedicated AI Savings Agent")
            chatbot_ui = gr.Chatbot(label="Savings Strategic Consultation Box", type="messages")
            user_msg = gr.Textbox(placeholder="Ask Advisor: 'Run tool' or 'Show my financial status'", label="Type Chat Message...")
            chat_send_btn = gr.Button("💬 Dispatch Query to Agent", variant="primary")

    # Wire up direct component pipelines
    apply_theme_btn.click(fn=apply_realtime_styles, inputs=[theme_selector], outputs=[style_injection_box, status_ticker])
    csv_btn.click(fn=process_and_index_savings_csv, inputs=[file_upload], outputs=[ledger_view, status_ticker])

    finalize_btn.click(
        fn=lambda sal, emg, has_sec, cost, s_date, df: calculate_savings_feasibility(
            sal, emg, has_sec, cost, s_date, float(df['Amount'].sum()) if df is not None and not df.empty else 0.0
        ),
        inputs=[user_salary, emg_target, has_secondary, sec_cost, sec_date, ledger_view],
        outputs=[report_box]
    )

    chat_send_btn.click(
        fn=savings_buddy_agent_response,
        inputs=[user_msg, chatbot_ui, user_name, user_curr, user_salary, emg_target, has_secondary, sec_cost, sec_date, ledger_view],
        outputs=[user_msg, chatbot_ui]
    )

app.launch(debug=True)

/tmp/ipykernel_1712/744410642.py:241: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot_ui = gr.Chatbot(label="Savings Strategic Consultation Box", type="messages")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b43ed05418c12892ce.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
